In [ ]:
import transformers

transformers.__version__

### sigclip

In [1]:
from PIL import Image
import requests
from transformers import AutoProcessor, AutoModel
import torch

model = AutoModel.from_pretrained("google/siglip-so400m-patch14-384")
processor = AutoProcessor.from_pretrained("google/siglip-so400m-patch14-384")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

texts = ["a photo of 2 cats", "a photo of 2 dogs"]
inputs = processor(text=texts, images=image, padding="max_length", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits_per_image = outputs.logits_per_image
probs = torch.sigmoid(logits_per_image) # these are the probabilities
print(f"{probs[0][0]:.1%} that image 0 is '{texts[0]}'")


/Users/zc478/miniconda3/envs/vis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/zc478/miniconda3/envs/vis/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/Users/zc478/miniconda3/envs/vis/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <5AA8DD3D-A2CC-31CA-8060-88B4E9C18B09> /Users/zc478/miniconda3/envs/vis/lib/python3.10/site-packages/torchvision/image.so
  Expected in:     <A69D69E1-75EF-32C4-83CE-D9457021DE94> /Users/zc478/miniconda3/envs/vis/lib/python3.10/site-packages/torch/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your

40.5% that image 0 is 'a photo of 2 cats'


### idefics2 torch

In [1]:
from transformers import Idefics2ForConditionalGeneration

model = Idefics2ForConditionalGeneration.from_pretrained("HuggingFaceM4/idefics2-8b")

/Users/zc478/miniconda3/envs/vis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 7/7 [01:17<00:00, 11.07s/it]


In [14]:
# single forward pass
input_ids = torch.LongTensor([[1,2,3]])
attention_mask = torch.ones_like(input_ids)
output = model.forward(input_ids=input_ids, attention_mask=attention_mask)

In [2]:
# [n for (n, p) in model.named_parameters()]

In [3]:
pt_vision_embeddings = model.model.vision_model.embeddings
vision_config = model.model.vision_model.config

In [4]:
import torch

In [5]:
patch_size = model.model.vision_model.config.patch_size
batch_size = 2
_unknown = 3
max_im_h = 224
max_im_w = 224

pixel_values = torch.ones(
    (batch_size, _unknown, max_im_h, max_im_w), requires_grad=False)
patch_attention_mask = torch.empty_like(
    pixel_values, requires_grad=False).fill_(True).to(torch.bool)
patch_attention_mask = torch.ones(
                (
                    batch_size,
                    pixel_values.size(2) // patch_size,
                    pixel_values.size(3) // patch_size,
                )
            )
patch_attention_mask = patch_attention_mask.to(dtype=torch.bool, device=pixel_values.device)

pt_embeddings_output = pt_vision_embeddings(pixel_values, patch_attention_mask)

In [7]:
import flax.linen as nn
import jax
import jax.numpy as jnp

from transformers.models.idefics2.configuration_idefics2 import \
    Idefics2VisionConfig

# from transformers.models.idefics2.modeling_flax_idefics2 import FlaxIdefics2VisionEmbeddings
# FlaxIdefics2VisionEmbeddings(model.model.vision_model.config)

In [8]:
# copied from FlaxCLIPVisionEmbeddings
class FlaxIdefics2VisionEmbeddings(nn.Module):
    config: Idefics2VisionConfig
    dtype: jnp.dtype = jnp.float32

    def setup(self):
        self.embed_dim = self.config.hidden_size
        self.image_size = self.config.image_size
        self.patch_size = self.config.patch_size

        # has weights and bias
        self.patch_embedding = nn.Conv(
            self.embed_dim,
            kernel_size=(self.patch_size, self.patch_size),
            strides=(self.patch_size, self.patch_size),
            padding="VALID",
            dtype=self.dtype,
            kernel_init=nn.initializers.normal(),
        )

        self.num_patches_per_side = self.image_size // self.patch_size
        self.num_patches = self.num_patches_per_side**2
        self.num_positions = self.num_patches
        # has weights
        self.position_embedding = nn.Embed(self.num_positions, self.embed_dim, embedding_init=nn.initializers.normal())
        self.position_ids = jnp.expand_dims(jnp.arange(0, self.num_positions, dtype="i4"), axis=0)

    def __call__(self, pixel_values: jnp.ndarray, patch_attention_mask: jnp.ndarray):
        embeddings = self.patch_embedding(pixel_values)
        batch_size, height, width, channels = embeddings.shape
        embeddings = jnp.reshape(embeddings, (batch_size, height * width, channels))
        boundaries = jnp.arange(1 / self.num_patches_per_side, 1.0, 1 / self.num_patches_per_side)

        def compute_position_ids(p_attn_mask):
            nb_patches_h = jnp.sum(p_attn_mask[:, 0])
            nb_patches_w = jnp.sum(p_attn_mask[0])

            fractional_coords_h = jnp.arange(0, 1 - 1e-6, 1 / nb_patches_h)
            fractional_coords_w = jnp.arange(0, 1 - 1e-6, 1 / nb_patches_w)

            bucket_coords_h = jnp.digitize(fractional_coords_h, boundaries, right=True)
            bucket_coords_w = jnp.digitize(fractional_coords_w, boundaries, right=True)

            pos_ids = (bucket_coords_h[:, None] * self.num_patches_per_side + bucket_coords_w).flatten()
            return pos_ids[jnp.ravel(p_attn_mask)]

        # (batch_size, max_nb_patches_h * max_nb_patches_w)
        position_ids = jax.vmap(compute_position_ids)(patch_attention_mask)
        embeddings = embeddings + self.position_embedding(position_ids)
        return embeddings


In [9]:
fx_vision_embeddings = FlaxIdefics2VisionEmbeddings(vision_config)

In [14]:
fx_pixel_values = jnp.ones(
    (batch_size, _unknown, max_im_h, max_im_w))
fx_patch_attention_mask = jnp.ones(
                (
                    batch_size,
                    pixel_values.size(2) // patch_size,
                    pixel_values.size(3) // patch_size,
                )
            , dtype=jnp.bool)

In [16]:
# fx_embeddings_output = fx_vision_embeddings.apply(fx_pixel_values, fx_patch_attention_mask)



AttributeError: 'jaxlib.xla_extension.ArrayImpl' object has no attribute 'items'

### mistral

In [1]:
import torch
import jax.numpy as jnp
from transformers import FlaxAutoModelForCausalLM
from transformers import MistralConfig
from transformers import FlaxMistralForCausalLM
from transformers import FlaxMistralModel
# config = MistralConfig.from_pretrained("mistralai/Mistral-7B-v0.1")
# model = FlaxAutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1", config=config, from_pt=True)
# model = FlaxMistralForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1", from_pt=True)
model = FlaxMistralForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1", dtype=jnp.bfloat16)

/Users/zc478/miniconda3/envs/vis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: Failed to import transformers.models.mistral.modeling_flax_mistral because of the following error (look up to see its traceback):
name 'FlaxMistralModel' is not defined

In [6]:
import jax.numpy as jnp
from transformers import AutoTokenizer, FlaxGemmaForCausalLM

model_id = "google/gemma-2b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"

model, params = FlaxGemmaForCausalLM.from_pretrained(
		model_id,
		dtype=jnp.bfloat16,
		revision="flax",
		_do_init=False,
)

inputs = tokenizer("Valencia and Málaga are", return_tensors="np", padding=True)
output = model.generate(**inputs, params=params, max_new_tokens=20, do_sample=False)
output_text = tokenizer.batch_decode(output.sequences, skip_special_tokens=True)

Gemma's activation function should be approximate GeLU and not exact GeLU. Changing the activation function to `gelu_pytorch_tanh`.if you want to use the legacy `gelu`, edit the `model.config` to set `hidden_activation=gelu`   instead of `hidden_act`. See https://github.com/huggingface/transformers/pull/29402 for more details.
Some of the weights of FlaxGemmaForCausalLM were initialized in bfloat16 precision from the model checkpoint at google/gemma-2b:
[('model', 'embed_tokens', 'embedding'), ('model', 'layers', '0', 'input_layernorm', 'weight'), ('model', 'layers', '0', 'mlp', 'down_proj', 'kernel'), ('model', 'layers', '0', 'mlp', 'gate_proj', 'kernel'), ('model', 'layers', '0', 'mlp', 'up_proj', 'kernel'), ('model', 'layers', '0', 'post_attention_layernorm', 'weight'), ('model', 'layers', '0', 'self_attn', 'k_proj', 'kernel'), ('model', 'layers', '0', 'self_attn', 'o_proj', 'kernel'), ('model', 'layers', '0', 'self_attn', 'q_proj', 'kernel'), ('model', 'layers', '0', 'self_attn', '

In [ ]:
# RUN_SLOW=1 RUN_PT_FLAX_CROSS_TESTS=1 pytest tests/models/mistral